# MAPED: experimental data on CUDA

Load your tilt series, align it step by step, and inspect the merged patterns.

## Setup

**Development prerequisite:** this workflow still requires MAPED development builds; the package command alone does not yet provide these APIs.

New to Python? Follow the [Conda or uv setup guide](https://github.com/bobleesj/quantem-tutorials/blob/oct-17-maped-tutorials/tutorials/diffraction/maped/setup/README.md). If packages are missing in an existing Python 3.11+ environment, uncomment the cell below, then restart the kernel.

In [ ]:
# %pip install --upgrade --pre -i https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple/ "quantem.widget[cuda]" quantem pandas

In [ ]:
from pathlib import Path

import quantem.widget as qw
from quantem.diffraction import MAPED
from quantem.gpu import io

folder = Path("tilt-series")
output = Path("outputs/merged_cuda_master.h5")
device = "cuda:0"
qw.profile()

## Load data

Put one Arina tilt series in `folder`, including master, data, and metadata files. Loading applies median replacement at stored bad detector pixels. Check that this correction and the recorded calibration are appropriate for your data.

In [ ]:
files = io.discover(folder)
tilts = io.load(
    files, device=device, apply_mask=False,
    hot_pixel_correction="median",
)
maped = MAPED.from_datasets(tilts)
maped.show_tilts(num_cols=7, size="medium")

## Align

`align_diffraction()` measures detector shifts; `align_images()` measures scan shifts from the default whole-detector mean images. Shifts use `(row, col)` pixels.

To use BF, ADF, DF or simultaneously recorded HAADF images, pass `images=images` to `align_images()`: one 2D array per tilt, with the same scan shape, sampling and orientation. The merge still uses the original diffraction data.

In [ ]:
maped.find_beam_center(show=False)
maped.align_diffraction(show=False)
maped.align_images(iterations=160, show=False)

In [ ]:
maped.show_alignment(num_cols=7, size="medium", reference=0)  # Set the zero-tilt index.

## Merge

Save scaled uint16 HDF5 and reopen its codes as ANS-resident data. Decode only requested regions, not the complete scan. Rounding is approximate: the summary reports RMS and maximum absolute error in counts, plus clipping.

In [ ]:
output.parent.mkdir(parents=True, exist_ok=True)
merged = maped.merge(crop=True, dtype="scaled_uint16", save_to=output, release_tilts=True, show=False)
maped.report()  # Table; use as_dict=True for a dictionary.

## Inspect results

Inspect merged diffraction patterns at three marked scan positions. Input tilts have been released to save memory. The examples below assume 512 × 512 input scans.

In [ ]:
maped.show(positions=[(128, 128), (256, 256), (384, 384)], tilt_indices=[])  # Merged patterns; inputs were released.

In [ ]:
qw.Show4DSTEM(merged, device=device)

## Reopen (optional)

The HDF5 result is already saved with its calibration and alignment metadata. Reopening restores calibrated intensities; it does not recover the original float32 values lost to uint16 rounding.

In [ ]:
# reopened = io.load(output, device=device)  # ANS resident.
# qw.Show4DSTEM(reopened, device=device)

Built on Henry Bell’s [QuantEM PR #256](https://github.com/electronmicroscopy/quantem/pull/256). Experimental inputs and executed outputs are not included in this shareable template.